# LiCONiC StoreX hello world

```{device-card} liconic-stx
```

| Property | Value |
|---|---|
| Manufacturer | LiCONiC |
| Device family | StoreX |
| Supported product line | STX |
| Connection | RS-232 serial |
| Serial settings | 9600 baud, 8 data bits, even parity, 1 stop bit, RTS/CTS |
| PyLabRobot class | `StoreX` |

STX, STR, and other names identify StoreX product lines. This driver accepts only known STX model identifiers because protocol compatibility with the other lines is not yet confirmed.

STX climate suffixes:

| Suffix | Climate type | Temperature | Active cooling | Independent humidity |
|---|---|---:|---:|---:|
| IC | Incubator | yes | no | no |
| HC | Humid cooler | yes | yes | no |
| DC2 | Dry storage | yes | no | yes |
| HR | Humid wide range | yes | yes | no |
| DR2 | Dry wide range | yes | no | yes |
| AR | Humidity controlled | yes | no | yes |
| DF | Deep freezer | yes | yes | no |
| NC | No climate | no | no | no |
| DH | Dry humid | yes | no | yes |

## Physical setup

1. Connect the controller's RS-232 interface to the computer.
2. Confirm the serial port name.
3. Install the physical cassettes in exactly the same order as the `racks` list below. Controller cassette numbering is 1-based.
4. Keep the loading tray and handler clear during setup and initialization.
5. Enable the optional shaker or barcode scanner only when that hardware is installed.

Cassette resources encode the plate pitch and number of sites. The configured list must match the physical instrument.

## Configure the device

This example uses an STX220 humidity-controlled variant. Replace the model, port, rack list, and loading-tray location with the physical configuration.

In [ ]:
from pylabrobot.liconic import StoreX
from pylabrobot.liconic.storex.racks import storex_rack_17mm_22, storex_rack_44mm_10
from pylabrobot.resources import Coordinate

racks = [
  storex_rack_44mm_10("cassette_1"),
  storex_rack_17mm_22("cassette_2"),
]

storex = StoreX(
  name="storex",
  model="STX220_AR",
  port="/dev/ttyUSB0",
  racks=racks,
  loading_tray_location=Coordinate.zero(),
  has_shaker=False,
)

## Connect

`setup()` opens the serial connection, performs the controller handshake, activates handling, and waits for the ready flag. It does not move a plate.

In [ ]:
await storex.setup()

## Initialize the handler

Initialization homes and activates the handler. Confirm the handler and transfer station are clear first.

In [ ]:
await storex.initialize()

## Store a plate

First represent the physical plate on the loading tray. This only updates PyLabRobot's resource state; place the real plate on the tray before continuing.

In [ ]:
from pylabrobot.resources import cor_96_wellplate_360uL_Fb

plate = cor_96_wellplate_360uL_Fb("assay_plate")
storex.loading_tray.assign_child_resource(plate)

Store the tray plate in the smallest available cassette site that fits it. Pass `"random"` or a specific `PlateHolder` to choose another destination.

In [ ]:
await storex.take_in_plate("smallest")

## Retrieve a plate

The plate is selected by its PyLabRobot resource name. Ensure the physical loading tray is empty.

In [ ]:
retrieved = await storex.fetch_plate_to_loading_tray("assay_plate")

## Move a plate internally

First put the plate back into a known source site.

In [ ]:
await storex.take_in_plate(racks[1].sites[0])

Move the stored plate to another free internal site. Resource state changes only after the controller completes the move.

In [ ]:
await storex.move_plate("assay_plate", racks[1].sites[1])

Inspect the current plate assignments for all configured cassettes.

In [ ]:
print(storex.summary())

## Barcode scanning

Pass a scanner with async `setup()`, `stop()`, and `scan_barcode()` methods as `barcode_scanner=` when constructing the StoreX. `KeyenceBarcodeScanner` is compatible when it matches the installed reader.

Scanning moves the selected site to the reading position, waits for the lift, triggers the scanner, resets the shovel, and stores the result on the plate resource.

In [ ]:
# barcode = await storex.scan_barcode(racks[1].sites[1])
# print(barcode)

## Temperature control

Every supported STX climate configuration except `_NC` accepts a temperature target in degrees Celsius.

In [ ]:
await storex.set_temperature(37.0)

Read the measured and target temperatures.

In [ ]:
current_temperature = await storex.request_current_temperature()
target_temperature = await storex.request_target_temperature()
print(current_temperature, target_temperature)

## Humidity control

The `_DC2`, `_DR2`, `_AR`, and `_DH` configurations support independent humidity control. Values are fractions from 0.0 to 1.0.

In [ ]:
await storex.set_humidity(0.90)

Read the measured and target relative-humidity fractions.

In [ ]:
current_humidity = await storex.request_current_humidity()
target_humidity = await storex.request_target_humidity()
print(current_humidity, target_humidity)

## CO₂ control

Use these operations only when CO₂ control hardware is installed. Values are fractions, so `0.05` means 5%.

In [ ]:
# await storex.set_co2_level(0.05)
# current_co2 = await storex.request_co2_level()
# target_co2 = await storex.request_target_co2_level()

## N₂ control

Use these operations only when N₂ control hardware is installed. Values are fractions.

In [ ]:
# await storex.set_n2_level(0.10)
# current_n2 = await storex.request_n2_level()
# target_n2 = await storex.request_target_n2_level()

## Shaking

Construct the StoreX with `has_shaker=True` only when the optional shaker is installed. Frequency is in hertz and must be between 1 and 50.

In [ ]:
# await storex.start_shaking(10.0)

Stop continuous shaking before accessing plates.

In [ ]:
# await storex.stop_shaking()

A timed shake uses client-side timing and sends the stop command when the wait exits.

In [ ]:
# await storex.shake(frequency=10.0, duration=30.0)

## Plate sensors

Read the shovel and primary transfer-station sensors when diagnosing plate presence.

In [ ]:
shovel_has_plate = await storex.request_shovel_sensor()
transfer_has_plate = await storex.request_transfer_sensor()
print(shovel_has_plate, transfer_has_plate)

Read the optional second transfer-station sensor separately.

In [ ]:
second_transfer_has_plate = await storex.request_second_transfer_sensor()

## Swap station

The position-specific methods read the current position and move only when needed.

In [ ]:
# await storex.move_swap_station_swapped()

Return the swap station home after the external transfer is complete.

In [ ]:
# await storex.move_swap_station_home()

## Door

Open the access door before external access.

In [ ]:
await storex.open_door()

After access is complete and the path is clear, close the door.

In [ ]:
await storex.close_door()

## Disconnect

Close the serial connections when finished.

In [ ]:
await storex.stop()